# Запуск в Colab

Включите GPU и загрузите архив проекта. Для Kaggle укажите путь в ARCHIVE.
Архив собирается из корня репозитория:

~~~bash
mkdir -p workspace/artifacts
rm -f workspace/artifacts/tfkp_setup.zip
zip workspace/artifacts/tfkp_setup.zip *.py *.json *.txt *.md *.jsonl *.csv support/*.py support/*.txt support/*.md support/*.ipynb workspace/textbook.pdf
~~~


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time
import urllib.request
import uuid
import zipfile

WORKSPACE = Path('/content' if Path('/content').exists() else '/kaggle/working')
ARCHIVE = WORKSPACE / 'tfkp_setup.zip'
if not ARCHIVE.exists() and Path('/content').exists():
    from google.colab import files
    uploaded = files.upload()
    ARCHIVE = Path(next(iter(uploaded))).resolve()
PROJECT = WORKSPACE / 'tfkp_project'
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as archive:
    archive.extractall(PROJECT)
SOLUTION = PROJECT
LOCAL = PROJECT / 'workspace'
LOCAL.mkdir(exist_ok=True)
PDF = LOCAL / 'textbook.pdf'
QA = PROJECT / 'support/tfkp_15_qa.md'
os.chdir(PROJECT)


## Окружение

Установка Python 3.11, зависимостей и загрузка моделей.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.9.5'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.11.14', 'workspace/.venv'], check=True)
PYTHON = str(LOCAL / '.venv/bin/python')
subprocess.run(['uv', 'pip', 'install', '--python', PYTHON, '-r', 'support/requirements-gpu.txt'], check=True)
subprocess.run([PYTHON, '-m', 'support.download_models', '--config', 'config.json', '--with-llm'], check=True)


## Данные и индекс

Подготовка корпуса и векторов E5 на CPU.


In [ ]:
subprocess.run([PYTHON, '-m', 'support.prepare_data', '--pdf', str(PDF), '--qa', str(QA), '--output', 'workspace/data'], check=True)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
subprocess.run([PYTHON, 'build_index.py', '--corpus', 'workspace/data/corpus.jsonl', '--output', 'workspace/index', '--config', 'config.json'], check=True)


## Qwen

Запуск модели на GPU. Журнал сохраняется в workspace/artifacts/server_stdout.log.


In [ ]:
runtime = json.loads((SOLUTION / 'runtime.json').read_text())
os.environ.update(LLM_BASE_URL='http://127.0.0.1:8000/v1', LLM_MODEL=runtime['model'],
                  LLM_API_KEY='local-only', SUBMISSION_ID='tfkp-' + uuid.uuid4().hex[:10])
(LOCAL / 'artifacts').mkdir(exist_ok=True)
server_log = open(LOCAL / 'artifacts/server_stdout.log', 'w')
server = subprocess.Popen([PYTHON, '-m', 'support.serve_qwen', '--config', 'config.json',
                           '--journal', 'workspace/artifacts/endpoint_log.jsonl'], stdout=server_log, stderr=subprocess.STDOUT)
started = time.monotonic()
while True:
    if server.poll() is not None:
        server_log.flush()
        raise RuntimeError((LOCAL / 'artifacts/server_stdout.log').read_text()[-12000:])
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=3) as response:
            health = json.load(response)
        if health.get('ready'):
            print(health)
            break
    except OSError:
        pass
    if time.monotonic() - started > 1200:
        raise TimeoutError('Проверьте workspace/artifacts/server_stdout.log')
    time.sleep(3)


## Эксперименты

Пять режимов с одинаковыми параметрами генерации.


In [ ]:
subprocess.run([PYTHON, '-m', 'support.run_experiments', '--index', 'workspace/index', '--questions', 'workspace/data/dev_questions.jsonl',
                '--gold', 'workspace/data/dev_gold.jsonl', '--config', 'config.json',
                '--endpoint-log', 'workspace/artifacts/endpoint_log.jsonl'], check=True)
print((SOLUTION / 'experiments.csv').read_text())
print((SOLUTION / 'report.md').read_text())


## Результаты

Сохранение ответов, метрик, отчета и журналов в ZIP.


In [ ]:
RESULT = WORKSPACE / 'tfkp_results.zip'
with zipfile.ZipFile(RESULT, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ('dev_answers.jsonl', 'dev_trace.jsonl', 'experiments.csv', 'report.md',
                 'own_questions.jsonl', 'own_gold.jsonl', 'config.json', 'runtime.json'):
        path = SOLUTION / name
        archive.write(path, path.relative_to(PROJECT))
    for path in (LOCAL / 'artifacts').rglob('*'):
        if path.is_file() and path.suffix in ('.json', '.jsonl'):
            archive.write(path, path.relative_to(PROJECT))
server.terminate()
server_log.close()
print(RESULT)
if Path('/content').exists():
    from google.colab import files
    files.download(str(RESULT))
